# 07 Model error

Does each design still image the BDT if the real velocity structure differs from the model? Echoes travel through a
perturbed "true" model; focusing uses the best-estimate 1D or 3D model. Same forward model, imaging and scoring as
05 (`mi_common.py`); only the echo travel times change.

Truth travel-time tables are on a coarse grid to keep them fast, so the variants are compared against a coarse-grid
`baseline` (the unchanged model on the coarse grid), not against 05. The LVZ scatterer strengths use the best model.
R is noise-free. P waves only.

Inputs: everything 05 reads. Outputs: `data/tt_var/`, `data/results_07_model_error.csv`

In [ ]:
# --- Imports ---
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skfmm
from scipy.ndimage import map_coordinates

from mi_common import MatrixImaging

In [ ]:
# --- Truth variants: how the real Earth might differ from the model ---
VARIANTS = {
    "baseline":                {},                    # unchanged model, coarse grid
    "all 5% faster":           dict(scale=1.05),      # whole model; no new tables needed
    "all 5% slower":           dict(scale=0.95),
    "shallow 10% faster":      dict(shallow=1.10),    # everything above the greywacke
    "shallow 10% slower":      dict(shallow=0.90),
    "clay-rich 50% thicker":   dict(clay=1.5),        # clay-rich base 50% deeper
    "greywacke 0.3 km higher": dict(gw=-0.3),         # greywacke top shallower everywhere
}

# --- Coarse grid for truth tables ---
CDXY, CDZ, CZMAX = 0.5, 0.2, 12.0      # km; coarse enough to be fast, fine enough for 2-6 Hz

# --- What to run ---
REFS_E = ["1d", "3d"]                  # focus with best 1D, or best 3D (fine tables from 03)
SHAPES_RUN = ["dome"]
DESIGNS = None                         # None = all, as in 05
RESULTS = "../data/results_07_model_error.csv"

mi = MatrixImaging("p")
VAR_DIR = f"{mi.DATA}/tt_var"
RUN = mi.scenarios(SHAPES_RUN)
slug = lambda s: re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")
print(len(VARIANTS), "variants x", len(RUN), "scenarios x", len(REFS_E), "focusing models")

In [ ]:
m = np.load(f"{mi.DATA}/velocity_model.npz")
x, y, z, vp0 = m["x"], m["y"], m["z"], m["vp"]
gw, cb, an = m["gw_top"][..., None], m["clay_base"][..., None], m["an_top"][..., None]
Z3 = z[None, None, :]
DZ = z[1] - z[0]


def column_value(vp, depth):
    """vp in each column at a given depth map (nearest cell)."""
    k = np.clip(np.round(depth / DZ).astype(int), 0, len(z) - 1)
    return np.take_along_axis(vp, k, axis=2)


def variant_model(v):
    """Best model with one change applied."""
    vp = vp0.copy()
    if "shallow" in v:
        vp = np.where(Z3 < gw, vp * v["shallow"], vp)
    if "clay" in v:
        new = np.minimum(cb * v["clay"], an)
        vp = np.where((Z3 >= cb) & (Z3 < new), column_value(vp0, np.full_like(cb, 0.2)), vp)
    if "gw" in v:
        new = gw + v["gw"]
        if v["gw"] < 0:      # raised: greywacke fills in
            vp = np.where((Z3 >= new) & (Z3 < gw), column_value(vp0, gw + DZ), vp)
        else:                # lowered: andesite fills in
            vp = np.where((Z3 >= gw) & (Z3 < new), column_value(vp0, gw - DZ), vp)
    return vp

In [ ]:
# --- Coarse truth tables for every station imaging uses (permanent, nodes, phase 2 bank) ---
step_xy, step_z = int(round(CDXY / (x[1] - x[0]))), int(round(CDZ / DZ))
kz = z <= CZMAX
xc, yc, zc_ = x[::step_xy], y[::step_xy], z[kz][::step_z]
XXc, YYc, ZZc = np.meshgrid(xc, yc, zc_, indexing="ij")
ixc, iyc = np.abs(xc) <= 13, np.abs(yc) <= 13
R0 = 0.6                               # starting sphere, larger on the coarse grid (km)


def travel_time_coarse(sx, sy, v):
    """P travel time (s) from a station to every coarse cell."""
    d = np.sqrt((XXc - sx) ** 2 + (YYc - sy) ** 2 + ZZc ** 2)
    v0 = v[np.argmin(abs(xc - sx)), np.argmin(abs(yc - sy)), 0]
    t = skfmm.travel_time(d - R0, v, dx=[CDXY, CDXY, CDZ], order=2)
    return np.where(d < R0, d / v0, t + R0 / v0)[ixc][:, iyc].astype(np.float16)


used = sorted(set(mi.PERMANENT) | set(mi.D["bank"]["ids"])
              | {s for ph in mi.D["designs"].values() for p in ph for s in p["stations"]})
for name, v in VARIANTS.items():
    if "scale" in v:
        continue                       # uses the baseline tables, rescaled
    folder = f"{VAR_DIR}/{slug(name)}"
    os.makedirs(folder, exist_ok=True)
    # contiguous copy: skfmm reads the raw memory, so a strided slice would be read in the wrong order
    vc = np.ascontiguousarray(variant_model(v)[::step_xy, ::step_xy][:, :, kz][:, :, ::step_z])
    for s in used:
        path = f"{folder}/{s:03d}_p.npy"
        if not os.path.exists(path):
            np.save(path, travel_time_coarse(mi.stations.x[s], mi.stations.y[s], vc))
    print(name, "tables done")

np.savez(f"{VAR_DIR}/grid.npz", x=xc[ixc], y=yc[iyc], z=zc_)

In [ ]:
# --- Echo travel times through a truth variant, at 05's scatterer candidates ---
gcv = np.load(f"{VAR_DIR}/grid.npz")
gc0 = np.array([gcv["x"][0], gcv["y"][0], gcv["z"][0]])
dgc = np.array([CDXY, CDXY, CDZ])
SC_IDX = ((np.column_stack([mi.tx[mi.SI], mi.ty[mi.SJ], mi.tz[mi.SK]]) - gc0) / dgc).T


def truth_times(variant):
    """tt_scat for MatrixImaging.forward: station -> travel times to every scatterer candidate in this variant."""
    v = VARIANTS[variant]
    folder = f"{VAR_DIR}/{slug('baseline' if 'scale' in v else variant)}"
    cache = {}

    def tt_scat(s):
        if s not in cache:
            T = map_coordinates(np.load(f"{folder}/{s:03d}_p.npy").astype(np.float32), SC_IDX, order=1)
            cache[s] = T / v.get("scale", 1.0)
        return cache[s]
    return tt_scat

In [ ]:
# --- Run: every variant x focusing model x scenario x design. Rerunning skips rows already in the results file ---
for variant in VARIANTS:
    tt_scat = truth_times(variant)
    for ref in REFS_E:
        err_res = mi.run(RUN, DESIGNS, results=RESULTS, extra=dict(variant=variant, ref=ref),
                         ref=ref, tt_scat=tt_scat, verbose=False)
    print(variant, "done", flush=True)
err_res = err_res[err_res.scenario.isin(RUN)]

In [ ]:
# --- Worst-case coverage for each variant and design ---
for ref in REFS_E:
    t = err_res[err_res.ref == ref].pivot_table(index="variant", columns="design", values="coverage", aggfunc="min")
    print(f"\nWorst-case coverage, focused with best {ref}")
    print(t.loc[list(VARIANTS)].round(2))

# --- Phase 1 location error for each variant ---
print("\nPhase 1 location error (km)")
p = err_res[(err_res.design == "phased (adaptive)") & (err_res.ref == REFS_E[0])]
print(p.groupby("variant").p1_error_km.agg(median="median", worst="max").loc[list(VARIANTS)].round(1))

In [ ]:
REF_SHOW = "1d"
SHOW_DESIGNS = [d for d in ["phased (adaptive)", "phased", "grid", "wide grid", "spiral"] if d in set(err_res.design)]
t = (err_res[err_res.ref == REF_SHOW]
     .pivot_table(index="variant", columns="design", values="coverage", aggfunc="min")
     .loc[list(VARIANTS), SHOW_DESIGNS])

with plt.style.context("default"):
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    im = ax.imshow(t.values, cmap="viridis", vmin=0, vmax=1, aspect="auto")
    for i in range(t.shape[0]):
        for j in range(t.shape[1]):
            ax.text(j, i, f"{t.values[i, j]:.2f}", ha="center", va="center",
                    color="w" if t.values[i, j] < 0.5 else "k", fontsize=8)
    ax.set(xticks=range(t.shape[1]), xticklabels=t.columns, yticks=range(t.shape[0]), yticklabels=t.index,
           title=f"Worst-case coverage, focused with best {REF_SHOW}")
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    fig.colorbar(im, label="Coverage of BDT top")
    plt.tight_layout()
    plt.show()